<img src="https://encrypted-tbn0.gstatic.com/images?q=tbn:ANd9GcSEaxoapR7GyB6FC7xp4aM-SrjhxTWTdFCmKAX3TvvFvQ&s=10" alt="Tecnológico de Costa Rica" width="160">

# 🌎 WALC 2026
### Ciencia de Datos · Introducción al Aprendizaje Automático
**Día 3 (Miércoles) — Clasificación 2 — Naive Bayes y regresión logística**

**Instituto Tecnológico de Costa Rica (TEC)** · Instructor: **Efrén Jiménez Delgado**
· 📧 efjimenez@itcr.ac.cr · 🔗 [LinkedIn](https://www.linkedin.com/in/ejimenezdelgado/)

# 📱 Naive Bayes y regresión logística: ¿en qué rango de precio cae un celular?

## 🧵 El hilo conductor (metodología CRISP-DM)

Este cuaderno sigue las seis fases de **CRISP-DM** [1] [2]: cada título de fase marca dónde estamos en el proceso.

| Fase | En este cuaderno |
|---|---|
| 🎯 Comprensión del negocio | Estimar el rango de precio (0 a 3) de un celular a partir de sus especificaciones. |
| 🔍 Comprensión de los datos | Cargamos 2 000 celulares con 20 variables y revisamos faltantes y distribución de clases. |
| 🧹 Preparación de los datos | Seleccionamos las 20 variables, dividimos 70/30 y estandarizamos usando solo el entrenamiento. |
| 🤖 Modelado | Entrenamos Naive Bayes gaussiano y, como refuerzo, regresión logística multiclase. |
| 📊 Evaluación | Matriz de confusión e informe por clase; comparamos ambos modelos. |
| 🚀 Despliegue y conclusiones | Interpretamos qué variables determinan el precio y qué modelo conviene. |

> 🎯 **Analogía:** un vendedor que mira las características de un celular (RAM, batería, cámara) y calcula qué tan probable es que pertenezca a cada rango de precio.

### 🗺️ Mapa del curso (cronograma Track 5 · Ciencia de datos)
| Día | Jornada | Teoría y laboratorio |
|---|---|---|
| 1 | Lunes | Predicción: regresión lineal y series de tiempo |
| 2 | Martes | Clasificación 1: árboles de decisión, bosques aleatorios y KNN |
| 👉 **3** | Miércoles | **Clasificación 2: SVM, regresión logística, Naive Bayes y redes neuronales** |
| 4 | Jueves | No supervisado: agrupación y asociación |
| 5 | Viernes | Ciencia de datos con IA generativa, ética y buenas prácticas |

---
## 🎯 Fase 1 · Comprensión del negocio

Estimar el rango de precio (0 a 3) de un celular a partir de sus especificaciones.

📚 Lectura recomendada: [3] [4]

## Análisis del Problema

Bob ha iniciado su propia empresa de telefonía móvil. Quiere dar una dura batalla a las grandes empresas como Apple, Samsung, etc.

No sabe cómo estimar el precio de los móviles que crea su empresa. En este competitivo mercado de la telefonía móvil, no se pueden simplemente asumir cosas. Para solucionar este problema, recopila datos de ventas de teléfonos móviles de varias empresas.

Bob quiere averiguar alguna relación entre las características de un teléfono móvil (por ejemplo: - RAM, memoria interna, etc.) y su precio de venta. Pero no es tan bueno en Machine Learning. Entonces necesita tu ayuda para resolver este problema.

En este problema, no tiene que predecir el precio real, sino un rango de precios que indica qué tan alto es el precio.

# Naive Bayes y regresión logística aplicados a precios de celulares

En este ejercicio se clasifica el **rango de precio** de un celular con **Naive Bayes**, un clasificador probabilístico basado en el teorema de Bayes, y se compara con la **regresión logística**.

### Autor
   * Efrén Jiménez

### Librerías

In [ ]:
import warnings
warnings.filterwarnings('ignore')
#imports
#numpy,pandas,scipy, math, matplotlib
import numpy as np
import pandas as pd
import scipy
from math import sqrt
import matplotlib.pyplot as plt
import seaborn as sns

#estimators

from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

#model metrics
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score
from sklearn.model_selection import cross_val_score
from sklearn.metrics import recall_score
from sklearn.metrics import precision_score
from sklearn import metrics

from sklearn.preprocessing import StandardScaler

#cross validation
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import confusion_matrix
from sklearn.metrics import roc_auc_score

import plotly.express as px
import plotly.figure_factory as ff


---
## 🔍 Fase 2 · Comprensión de los datos

Cargamos 2 000 celulares con 20 variables y revisamos faltantes y distribución de clases.

📚 Lectura recomendada: [7]

## Entendimiento de los Datos

El total es de 2000 registro con 21 columnas dentro del set de datos de entrenamiento

- battery_power,
- blue,
- clock_speed,
- dual_sim,
- fc,
- four_g,
- int_memory,
- m_dep,
- mobile_wt,
- n_cores,
- pc,
- px_height
- ,px_width
- ,ram
- ,sc_h
- ,sc_w,
- talk_time,
- three_g,
- touch_screen,
- wifi,
- price_range

In [ ]:
import os, urllib.request

os.makedirs('sample_data', exist_ok=True)
ruta = 'sample_data/celulares_train.csv'
if not os.path.exists(ruta):
    # En el repositorio hay dos archivos llamados train.csv (Titanic en dia_3, celulares en dia_4); este se guarda con otro nombre
    urllib.request.urlretrieve('https://gitlab.com/ejimenez/fiec_intro_machine_learning/-/raw/main/dia_4/train.csv', ruta)
print('Datos de celulares listos:', ruta)

In [ ]:
#Cargar los datos
train = pd.read_csv(r"sample_data/celulares_train.csv")

In [ ]:
#Primeros registros
train.head()

In [ ]:
#Describir dataset
train.describe()

In [ ]:
#Información del dataset
train.info()

In [ ]:
# Numero de registros del dataset
print("Cantidad de registros:",len(train))
# Cantidad de columnas del dataset
print("Cantidad de columnas:",len(train.columns))
# Tipos de datos del dataset
print("Tipo de datos:",train.dtypes.unique())
# Tamaño del daset
print("Filas y columnas:",train.shape)

In [ ]:
# Chekear valores nulos
print(train.isnull().any().sum(), ' / ', len(train.columns))

In [ ]:
#Cuales columnnas estas con valores nulos
train.isnull().sum()[train.isnull().sum() > 0]

In [ ]:
# N/A= Valores vacios
#Usar solo si se quiere eliminar los registros que contienen columnas vacías
train=train.dropna()

In [ ]:
train["price_range"].unique()

In [ ]:
train['price_range'].value_counts(normalize=True)

In [ ]:
train['price_range'].value_counts()

In [ ]:
fig = px.histogram(train, x="price_range", nbins=4, color="price_range")
fig.update_layout(title_text="Distribución del rango de precio en el Dataset")
fig.show()

## Exploración de los Datos

#### Gráficos exploratorios

In [ ]:
plt.figure(1, figsize=(15, 6))
n = 0
for x in ['battery_power', 'wifi', 'ram']:
    n += 1
    plt.subplot(1, 3, n)
    sns.histplot(train[x], bins=20, kde=True)
    plt.title('Distribución de {}'.format(x))
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(1 , figsize = (15 , 5))
sns.countplot(y = 'price_range' , data = train)
plt.show()

In [ ]:
plt.figure(1 , figsize = (15 , 7))
n = 0
for x in ['battery_power' , 'wifi' , 'ram']:
    for y in ['battery_power' , 'wifi' , 'ram']:
        n += 1
        plt.subplot(3 , 3 , n)
        plt.subplots_adjust(hspace = 0.5 , wspace = 0.5)
        sns.regplot(x = x , y = y , data = train)
        plt.ylabel(y.split()[0]+' '+y.split()[1] if len(y.split()) > 1 else y )
plt.show()

In [ ]:
labels = train.n_cores.value_counts().index
colors = ['green','blue','red','yellow','orange','gray','lime','pink']
explode = [0,0,0,0,0,0,0,0,]
sizes = train.n_cores.value_counts().values

# visual
plt.figure(figsize = (7,7))
plt.pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%')
plt.title('Distribucion de los nucleos de los celulares',color = 'blue',fontsize = 15)

In [ ]:
sns.boxplot(x="price_range", y="int_memory", hue="price_range", data=train, palette="PRGn")
plt.show()

In [ ]:
sns.swarmplot(x="price_range", y="clock_speed",hue="price_range", data=train)
plt.show()

In [ ]:
sns.swarmplot(x="price_range", y="dual_sim",hue="price_range", data=train)
plt.show()

---
## 🧹 Fase 3 · Preparación de los datos

Seleccionamos las 20 variables, dividimos 70/30 y estandarizamos usando solo el entrenamiento.

📚 Lectura recomendada: [8]

Dividir el conjunto de datos en uno de entrenamiento y otro de pruebas:

In [ ]:
X = train[["battery_power","blue","clock_speed","dual_sim","fc","four_g","int_memory",
           "m_dep","mobile_wt","n_cores","pc","px_height","px_width","ram","sc_h","sc_w",
           "talk_time","three_g","touch_screen","wifi"]]

y = train["price_range"]

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=1234, stratify=y)

# Se estandariza con la media y desviación del ENTRENAMIENTO únicamente (evita fuga de información)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

>Es importante siempre validar los rangos de los conjuntos de datos creados, para evitar caer en extrapolación:

---
## 🤖 Fase 4 · Modelado

Entrenamos Naive Bayes gaussiano y, como refuerzo, regresión logística multiclase.

📚 Lectura recomendada: [4] [5]

## Modelo de Machine Learning

**Naive Bayes** aplica el teorema de Bayes, $P(y\mid x)\propto P(y)\prod_j P(x_j\mid y)$, suponiendo que las variables son **independientes dentro de cada clase** (de ahí lo "ingenuo"). En la versión *gaussiana* cada variable se modela con una distribución normal por clase [3] [4].

![Naive Bayes gaussiano](https://raw.githubusercontent.com/jakevdp/PythonDataScienceHandbook/master/notebooks/figures/05.05-gaussian-NB.png)
*Figura: cada clase se resume con una distribución normal por variable (elipses); un punto nuevo se asigna a la clase más probable. [11]*

Una vez seleccionadas las variables para incluir en el modelo de clasificación, se procede a crearlo:

In [ ]:
nb = GaussianNB()

In [ ]:
# Entrenamos nuestro modelo
nb.fit(X_train, y_train)

In [ ]:
# Hacemos las predicciones que en definitiva una línea (en este caso, al ser 2D)
y_pred1 = nb.predict( X_test )

In [ ]:
y_pred1

In [ ]:
predict_values=nb.predict_proba(X_test)
predict_values[:, 1]

---
## 📊 Fase 5 · Evaluación

Matriz de confusión e informe por clase; comparamos ambos modelos.

📚 Lectura recomendada: [9]

In [ ]:
np.array(y_pred1)

In [ ]:
matrix=confusion_matrix(np.array(y_test), np.array(y_pred1))
matrix

In [ ]:
print(classification_report(np.array(y_test), np.array(y_pred1)))

In [ ]:
figure = plt.figure(figsize=(8, 8))
sns.heatmap(matrix, annot=True, fmt='g', cmap=plt.cm.Blues)
plt.tight_layout()
plt.ylabel('Rango real')
plt.xlabel('Rango predicho')
plt.show()

---
## 🚀 Fase 6 · Despliegue y conclusiones

Interpretamos qué variables determinan el precio y qué modelo conviene.

📚 Lectura recomendada: [6]

## Resultados

Naive Bayes clasifica correctamente el rango de precio en **≈ 79 %** de los celulares de prueba (la clase 0, la más barata, y la 3, la más cara, se reconocen mejor que las intermedias, que se confunden entre sí). En el refuerzo, la **regresión logística** llega a **≈ 95 %**: la `ram` casi separa por sí sola los rangos de precio y la regresión logística aprovecha mejor esa información que el supuesto de independencia de Naive Bayes.

---
## 🧪 Refuerzo: regresión logística: de la probabilidad a la clase

La **regresión logística** modela directamente la probabilidad de cada clase. Para dos clases calcula una combinación lineal de las variables, $z = \beta_0 + \beta_1 x_1 + \dots + \beta_p x_p$, y la convierte en probabilidad con la **función sigmoide** $P(y=1\mid x)=\dfrac{1}{1+e^{-z}}$ [5]. Con más de dos clases (como los 4 rangos de precio) usa la generalización *softmax*. Se interpreta con **razones de momios** (*odds ratio*): $e^{\beta_j}$ indica cuánto se multiplican los momios de pertenecer a la clase por cada aumento de una desviación estándar de la variable [5].

![Función sigmoide](https://raw.githubusercontent.com/rasbt/python-machine-learning-book-3rd-edition/master/ch03/images/03_02.png)
*Figura: la sigmoide convierte cualquier valor z en una probabilidad entre 0 y 1. [10]*

A diferencia de Naive Bayes (que asume variables independientes dentro de cada clase), la regresión logística **aprende los pesos** de las variables juntas y por eso suele ser más precisa cuando las variables están correlacionadas [4].

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score

logit = LogisticRegression(max_iter=2000, C=1.0)
logit.fit(X_train, y_train)
pred_lr = logit.predict(X_test)

print(f'Exactitud  Naive Bayes (GaussianNB)   : {accuracy_score(y_test, y_pred1):.3f}')
print(f'Exactitud  Regresión logística        : {accuracy_score(y_test, pred_lr):.3f}')
print(f'AUC macro (uno-contra-resto) NB       : {roc_auc_score(y_test, nb.predict_proba(X_test), multi_class="ovr"):.3f}')
print(f'AUC macro (uno-contra-resto) logística: {roc_auc_score(y_test, logit.predict_proba(X_test), multi_class="ovr"):.3f}')

# Variables que más empujan hacia el rango de precio más alto (clase 3): razón de momios por desviación estándar
columnas = ['battery_power','blue','clock_speed','dual_sim','fc','four_g','int_memory','m_dep','mobile_wt','n_cores',
            'pc','px_height','px_width','ram','sc_h','sc_w','talk_time','three_g','touch_screen','wifi']
momios = pd.Series(np.exp(logit.coef_[3]), index=columnas).sort_values(ascending=False)
print('\nRazón de momios (clase 3 = precio muy alto) por cada +1 desv. estándar:')
print(momios.head(5).round(2))

### ✍️ Ejercicios
1. ¿Cuál modelo acierta más? ¿Por qué la regresión logística aprovecha mejor variables correlacionadas como `px_height` y `px_width`?
2. Pruebe `C` = 0.01, 0.1, 1, 10: ¿qué le pasa a los coeficientes y a la exactitud (regularización L2)?
3. ¿Qué variable domina el precio? Compare con el gráfico de caja de `ram` por rango de precio.

---
## ✅ Conclusiones y siguientes pasos

Naive Bayes es muy rápido y funciona con pocos datos, pero su supuesto de independencia rara vez se cumple del todo [4]; la regresión logística, que estima los pesos de todas las variables a la vez, suele ser más precisa y además entrega **razones de momios** interpretables [5]. Para decidir, compare ambos con validación cruzada y las métricas de cada clase [9].

---
## 📚 Referencias

Fuentes adicionales para profundizar: [12].

**[1]** Chapman, P., Clinton, J., Kerber, R., Khabaza, T., Reinartz, T., Shearer, C., & Wirth, R. (2000). *CRISP-DM 1.0: Step-by-step data mining guide*. SPSS Inc. https://www.the-modeling-agency.com/crisp-dm.pdf

**[2]** Wirth, R., & Hipp, J. (2000). CRISP-DM: Towards a standard process model for data mining. *Proceedings of the 4th International Conference on the Practical Applications of Knowledge Discovery and Data Mining*, 29–39.

**[3]** Maron, M. E. (1961). Automatic indexing: An experimental inquiry. *Journal of the ACM, 8*(3), 404–417. https://doi.org/10.1145/321075.321084

**[4]** Domingos, P., & Pazzani, M. (1997). On the optimality of the simple Bayesian classifier under zero-one loss. *Machine Learning, 29*(2–3), 103–130. https://doi.org/10.1023/A:1007413511361

**[5]** Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X. (2013). *Applied Logistic Regression* (3.ª ed.). Wiley. https://doi.org/10.1002/9781118548387

**[6]** Murphy, K. P. (2012). *Machine Learning: A Probabilistic Perspective*. MIT Press.

**[7]** Sharma, A. (2017). *Mobile Price Classification* [Conjunto de datos]. Kaggle. https://www.kaggle.com/datasets/iabhishekofficial/mobile-price-classification

**[8]** Pedregosa, F., Varoquaux, G., Gramfort, A., Michel, V., Thirion, B., Grisel, O., … Duchesnay, E. (2011). Scikit-learn: Machine learning in Python. *Journal of Machine Learning Research, 12*, 2825–2830. https://jmlr.org/papers/v12/pedregosa11a.html

**[9]** Powers, D. M. W. (2011). Evaluation: From precision, recall and F-measure to ROC, informedness, markedness and correlation. *Journal of Machine Learning Technologies, 2*(1), 37–63.

**[10]** Raschka, S., & Mirjalili, V. (2019). *Python Machine Learning* (3.ª ed.). Packt Publishing. (Figuras usadas desde https://github.com/rasbt/python-machine-learning-book-3rd-edition)

**[11]** VanderPlas, J. (2016). *Python Data Science Handbook*. O'Reilly Media. https://jakevdp.github.io/PythonDataScienceHandbook/ (figuras usadas desde https://github.com/jakevdp/PythonDataScienceHandbook)

**[12]** McKinney, W. (2010). Data structures for statistical computing in Python. *Proceedings of the 9th Python in Science Conference*, 56–61. https://doi.org/10.25080/Majora-92bf1922-00a